In [ ]:
%load_ext autoreload
%autoreload 2
import pandas as pd
from pathlib import Path
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats
from pipeline.utils import enrichment

file_path = "/home/sjkim/protocols/il_11/exon.tsv"
prefix = ["Ctrl", "IL11_24h", ]

control, test = prefix

df = pd.read_csv(file_path, sep="\t", comment="#")
df = df.drop(columns=["Chr", "Start", "End", "Strand", "Length"]).set_index("Geneid")
df.columns = [Path(x).stem for x in df.columns]

groups = {
    s: next((p for p in prefix if s.startswith(p)), None)
    for s in df.columns
}
samples = [s for s in df.columns if groups[s] is not None]
counts = df[samples].T.astype(int)

metadata = pd.DataFrame(
    {"condition": [groups[s] for s in samples]},
    index=samples,
)

dds = DeseqDataSet(
    counts=counts,
    metadata=metadata,
    design="~condition",
)
dds.deseq2()

stats = DeseqStats(dds, contrast=["condition", test, control])
stats.summary()
deg = stats.results_df.copy()

coeff = f"condition[T.{test}]"
stats.lfc_shrink(coeff=coeff)

deg["log2FoldChange_shrunk"] = stats.results_df["log2FoldChange"]
deg["lfcSE_shrunk"] = stats.results_df["lfcSE"]
deg["contrast"] = f"{test}_vs_{control}"

gsea_result = enrichment.gsea(deg, species="mouse", series="il_11")

deg.to_csv(f"{test}_vs_{control}_deg.tsv", sep="\t", index=True)
gsea_result.to_csv(f"{test}_vs_{control}_gsea.tsv", sep="\t", index=False)